<a href="https://colab.research.google.com/github/Razquin-21/DeepLearning/blob/main/assignment_2/modelos_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Configuración de Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# 2. Preparar los Datos (Aquí definimos train_loader)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Cargando dataset de Flowers102...")
train_dataset = torchvision.datasets.Flowers102(root='./data', split='train', download=True, transform=transform)
test_dataset = torchvision.datasets.Flowers102(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print("Datos listos.")

# 3. Diseñar la arquitectura CNN desde cero (AlexNet)
class SimpleAlexNet(nn.Module):
    def __init__(self, num_classes=102):
        super(SimpleAlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model_scratch = SimpleAlexNet(num_classes=102).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.Adam(model_scratch.parameters(), lr=0.001)

# 4. Inicio del Entrenamiento
num_epochs = 10
print("Iniciando el entrenamiento desde cero (AlexNet-like)... Esto tomará unos minutos.")

for epoch in range(num_epochs):
    model_scratch.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)
        optimizer_scratch.zero_grad()
        outputs = model_scratch(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_scratch.step()
        running_loss += loss.item()

    print(f'Epoca [{epoch+1}/{num_epochs}], Perdida: {running_loss / len(train_loader):.4f}')

print("Entrenamiento del Modelo 1 completado")

Usando dispositivo: cuda
Cargando dataset de Flowers102...


100%|██████████| 345M/345M [00:13<00:00, 26.3MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.60MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 45.6MB/s]


Datos listos.
Iniciando el entrenamiento desde cero (AlexNet-like)... Esto tomará unos minutos.
Época [1/10], Pérdida: 4.7294
Época [2/10], Pérdida: 4.6266
Época [3/10], Pérdida: 4.6155
Época [4/10], Pérdida: 4.5843
Época [5/10], Pérdida: 4.6047
Época [6/10], Pérdida: 4.5505
Época [7/10], Pérdida: 4.4874
Época [8/10], Pérdida: 4.3611
Época [9/10], Pérdida: 4.2062
Época [10/10], Pérdida: 4.0894
¡Entrenamiento del Modelo 1 completado!


In [7]:
# 1. Cargar el modelo preentrenado (VGG16)
print("Descargando modelo VGG16 preentrenado...")
model_transfer = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)

# 2. Congelar las capas convolucionales (Transfer Learning)
for param in model_transfer.parameters():
    param.requires_grad = False

# 3. Adaptar la última capa (El clasificador)
# Reemplazamos la última capa lineal para que tenga 102 salidas (nuestras flores) en lugar de las 1000 originales de VGG
num_features = model_transfer.classifier[6].in_features
model_transfer.classifier[6] = nn.Linear(num_features, 102)

# Enviar el modelo modificado a la GPU
model_transfer = model_transfer.to(device)

# 4. Definir función de pérdida y optimizador (Solo optimizamos la capa nueva)
criterion_transfer = nn.CrossEntropyLoss()
# Solo pasamos los parámetros del classifier a Adam, no toda la red
optimizer_transfer = optim.Adam(model_transfer.classifier.parameters(), lr=0.001)

# 5. Bucle de Entrenamiento (Transfer Learning)
num_epochs_transfer = 5 # El Transfer Learning necesita menos epocas porque ya sabe mucho

print("Iniciando Transfer Learning con VGG16... Esto será más rápido.")

for epoch in range(num_epochs_transfer):
    model_transfer.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer_transfer.zero_grad()

        outputs = model_transfer(inputs)
        loss = criterion_transfer(outputs, labels)

        loss.backward()
        optimizer_transfer.step()

        running_loss += loss.item()

    print(f'Epoca [{epoch+1}/{num_epochs_transfer}], Perdida: {running_loss / len(train_loader):.4f}')

print("Transfer Learning completado")

Descargando modelo VGG16 preentrenado...
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 140MB/s]


Iniciando Transfer Learning con VGG16... Esto será más rápido.
Época [1/5], Pérdida: 3.8048
Época [2/5], Pérdida: 1.3673
Época [3/5], Pérdida: 0.7771
Época [4/5], Pérdida: 0.5276
Época [5/5], Pérdida: 0.4126
¡Transfer Learning completado!
